# Week 8：K-Means 簇解释与局限性

目标：把 cluster id 转换成可以描述的样本群体，并明确 K-Means 适用与不适用的情形。

## 1. 聚类后的正确工作

K-Means 完成后，不能停在“cluster 0 / 1 / 2”。要查看每一簇在原始特征上的均值或中位数，才能形成可解释的画像：

- 哪些特征在这一群体中明显更高或更低？
- 这个群体可能对应怎样的用户、商品或样本？
- 这些差异是否稳定、是否能支持后续行动？

注意：画像是基于数据的解释，不等于因果结论。

In [1]:
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

In [2]:
dataset = load_iris(as_frame=True)
X = dataset.data

# K-Means 基于距离，因此先标准化。
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, n_init=20, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

clustered_data = X.copy()
clustered_data['cluster'] = clusters
display(clustered_data.head())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),cluster
0,5.1,3.5,1.4,0.2,1
1,4.9,3.0,1.4,0.2,1
2,4.7,3.2,1.3,0.2,1
3,4.6,3.1,1.5,0.2,1
4,5.0,3.6,1.4,0.2,1


In [3]:
# 每簇的原始特征平均值：这是业务解释时最直观的表。
profile = clustered_data.groupby('cluster').mean().round(2)
profile['sample_count'] = clustered_data.groupby('cluster').size()
display(profile)

# K-Means 的中心原本位于标准化空间；inverse_transform 将它还原到原始特征单位，便于核对。
centers_original = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=X.columns,
    index=pd.Index(range(3), name='cluster'),
).round(2)
print('还原到原始单位的聚类中心：')
display(centers_original)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),sample_count
cluster,,,,,
0,5.80,2.67,4.37,1.41,53
1,5.01,3.43,1.46,0.25,50
2,6.78,3.10,5.51,1.97,47


还原到原始单位的聚类中心：


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
cluster,,,,
0,5.80,2.67,4.37,1.41
1,5.01,3.43,1.46,0.25
2,6.78,3.10,5.51,1.97


## 2. 解释簇的一个例子

例如，若某簇的花瓣长度和花瓣宽度均值明显更小，可以把它暂时描述为“花瓣较小群体”。这只是特征画像，不应该直接写成某个真实物种名称；真实标签若存在，只能在聚类结束后用于验证或帮助命名。

## 3. K-Means 的局限

K-Means 比较适合大小相近、近似圆形或球形、密度相近的簇。它可能表现不佳的情况：

- 簇是弯月形、环形等不规则形状。
- 簇大小或密度差异很大。
- 异常值很多；均值容易被极端值拉偏。
- 不知道合理的 K，且没有可解释的业务分组目标。

此时可以考虑 DBSCAN、层次聚类，或先处理异常值；具体选择仍须根据数据与任务验证。

思考题：为什么异常值会特别影响 K-Means 的结果？